# Phase 1 — Data Preparation, Chunking & Embeddings

**Project:** Last-Mile Delivery RAG Assistant  
**Author:** Everton Gomes  
**Phase goal:** Understand how raw documents become searchable vector representations.

---

## What we're building in this notebook

Before an LLM can answer questions about our delivery operations documents, we need to:

1. **Load** the raw documents (Markdown files)
2. **Chunk** them — split into smaller pieces that fit inside an LLM's context window
3. **Embed** each chunk — convert text into a dense vector of numbers
4. **Index** those vectors in a vector store (FAISS, then ChromaDB)
5. **Query** the index to confirm retrieval works

Each step has a 'Why this matters' explanation so you understand the concept, not just the code.

---

## Prerequisites

```bash
pip install langchain langchain-community sentence-transformers faiss-cpu chromadb
```

## Step 0 — Imports and setup

We import everything upfront so it's clear what each library's role is:

| Library | Role |
|---|---|
| `langchain` | Orchestration: loaders, splitters, chain building |
| `sentence_transformers` | Converts text → dense vectors (embeddings) |
| `faiss` | Fast vector similarity search (in-memory) |
| `chromadb` | Persistent vector database |
| `pathlib` | Clean file path handling |
| `time` | Benchmarking retrieval latency |

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import time
import json
from pathlib import Path

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

# ── Embeddings ────────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np

# ── Vector stores ─────────────────────────────────────────────────────────────
import faiss
import chromadb
from chromadb.config import Settings

# ── Utilities ─────────────────────────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Imports OK")
print(f"Working directory: {Path.cwd()}")

✅ Imports OK
Working directory: /Users/evertongomes/lastmile-delivery-rag/notebooks


---

## Step 1 — Load the documents

### Why this matters

LangChain's `DirectoryLoader` reads all files in a folder and wraps each in a
`Document` object. A `Document` has two parts:

- **`page_content`**: the raw text
- **`metadata`**: a dict with source file path, page number, etc.

Metadata is crucial — it's how we'll later tell the user *which document*
the answer came from (source attribution in the Streamlit UI).

**Key concept:** Loading is just reading. We haven't split or embedded anything yet.

In [2]:
DATA_DIR = Path("../data/raw")

# DirectoryLoader walks the folder and applies TextLoader to each .md file
loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.md",           # match all markdown files recursively
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

documents = loader.load()

print(f"\n📄 Loaded {len(documents)} documents")
print("\nDocument overview:")
for doc in documents:
    source = Path(doc.metadata["source"]).name
    char_count = len(doc.page_content)
    word_count = len(doc.page_content.split())
    print(f"  • {source:<35} {char_count:>6} chars  |  {word_count:>5} words")

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 1504.74it/s]


📄 Loaded 5 documents

Document overview:
  • capacity_planning.md                  3728 chars  |    586 words
  • routing_rules.md                      4141 chars  |    694 words
  • zone_definitions.md                   3166 chars  |    536 words
  • station_operations.md                 4212 chars  |    632 words
  • sla_policies.md                       3399 chars  |    536 words


In [3]:
# Inspect a document — always do this before processing
sample_doc = documents[0]

print("=== METADATA ===")
print(sample_doc.metadata)

print("\n=== FIRST 500 CHARS OF CONTENT ===")
print(sample_doc.page_content[:500])

=== METADATA ===
{'source': '../data/raw/capacity_planning.md'}

=== FIRST 500 CHARS OF CONTENT ===
# Meridian Logistics — Capacity Planning & Fleet Management Guidelines (Synthetic)

> **Disclaimer:** Synthetic document for educational/portfolio purposes only.

---

## 1. Capacity Planning Cycle

Meridian Logistics uses a rolling 13-week capacity planning cycle, updated every Monday.

### Planning Horizons
- **Week 1–2 (Operational):** Firm plan. Fleet and headcount locked.
  Changes require Station Manager approval.
- **Week 3–6 (Tactical):** Provisional plan. Volume forecasts drive fleet
  booking; ven


---

## Step 2 — Chunking strategy experiments

### Why this matters — the chunking problem

An LLM's context window is limited (e.g. ~4k–128k tokens). We cannot feed
entire documents to the model — we need to find and inject only the *relevant
pieces*. Those pieces are called **chunks**.

But there's a trade-off:

| Chunk size | Retrieval precision | Risk |
|---|---|---|
| Very small (100–200 chars) | High — very targeted | Context is lost; the chunk may be meaningless without surrounding text |
| Medium (400–800 chars) | Balanced | Good default starting point |
| Very large (1500+ chars) | Low — may include irrelevant text | LLM context bloat; harder to retrieve precisely |

**Overlap** (`chunk_overlap`) means consecutive chunks share some text.
This prevents a key fact at the boundary between two chunks from being
split and lost.

### We'll compare three strategies:
1. **Fixed-size** (character count) — simplest, no linguistic awareness
2. **Recursive character** — tries to split on natural boundaries (paragraphs → sentences → words)
3. **Manual inspection** — visualise the actual chunks produced

In [4]:
# ── Strategy A: Fixed-size character splitting ─────────────────────────────────
# Splits every N characters regardless of sentence or paragraph boundaries.
# Simplest approach — useful baseline but often cuts sentences mid-way.

fixed_splitter = CharacterTextSplitter(
    chunk_size=500,       # each chunk = ~500 characters
    chunk_overlap=50,     # 50-char overlap between consecutive chunks
    separator="\n",       # try to split on newlines first
)

fixed_chunks = fixed_splitter.split_documents(documents)

print(f"Strategy A (fixed, 500 chars): {len(fixed_chunks)} chunks")
print(f"  Avg chunk size: {sum(len(c.page_content) for c in fixed_chunks) / len(fixed_chunks):.0f} chars")

Strategy A (fixed, 500 chars): 42 chunks
  Avg chunk size: 449 chars


In [5]:
# ── Strategy B: Recursive character splitting ──────────────────────────────────
# This is the recommended default in LangChain.
# It tries to split on ["\n\n", "\n", " ", ""] in order —
# meaning it respects paragraph breaks first, then line breaks, then words.
# Much less likely to cut a sentence in the middle.

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""],  # priority order
)

recursive_chunks = recursive_splitter.split_documents(documents)

print(f"Strategy B (recursive, 500 chars): {len(recursive_chunks)} chunks")
print(f"  Avg chunk size: {sum(len(c.page_content) for c in recursive_chunks) / len(recursive_chunks):.0f} chars")

Strategy B (recursive, 500 chars): 55 chunks
  Avg chunk size: 346 chars


In [6]:
# ── Strategy C: Larger chunks with more overlap ────────────────────────────────
# Useful when context richness matters more than retrieval precision.
# Good for complex multi-step policies (like our SLA and routing docs).

large_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""],
)

large_chunks = large_splitter.split_documents(documents)

print(f"Strategy C (large, 1000 chars):    {len(large_chunks)} chunks")
print(f"  Avg chunk size: {sum(len(c.page_content) for c in large_chunks) / len(large_chunks):.0f} chars")

Strategy C (large, 1000 chars):    23 chunks
  Avg chunk size: 851 chars


In [8]:
# ── Visual comparison — look at the same section across strategies ─────────────
# Let's look at how the SLA policy document gets chunked differently

def get_chunks_from_doc(chunks, filename_keyword, n=3):
    """Return first n chunks from a specific source document."""
    return [
        c for c in chunks
        if filename_keyword in c.metadata.get("source", "")
    ][:n]

print("=" * 70)
print("SAME DOCUMENT (sla_policies.md) — FIRST 2 CHUNKS PER STRATEGY")
print("=" * 70)

for label, chunks in [
    ("A — Fixed 500", fixed_chunks),
    ("B — Recursive 500", recursive_chunks),
    ("C — Recursive 1000", large_chunks),
]:
    selected = get_chunks_from_doc(chunks, "sla_policies")
    print(f"\n── Strategy {label} ({len(get_chunks_from_doc(chunks, 'sla_policies', 999))} total chunks from this doc) ──")
    for i, chunk in enumerate(selected):
        print(f"  [Chunk {i+1}] {len(chunk.page_content)} chars")
        print(f"  {chunk.page_content[:200].strip()}...")
        print()

SAME DOCUMENT (sla_policies.md) — FIRST 2 CHUNKS PER STRATEGY

── Strategy A — Fixed 500 (8 total chunks from this doc) ──
  [Chunk 1] 437 chars
  # Meridian Logistics — Service Level Agreement (SLA) Policies (Synthetic)
> **Disclaimer:** Synthetic document for educational/portfolio purposes only.
---
## 1. Service Tiers
Meridian Logistics operates three cu...

  [Chunk 2] 485 chars
  - **Failure threshold:** SLA breach if delivery occurs after 10:30.
- **Compensation:** Full refund of surcharge on breach + credit note of €2.00.
### Express (EX)
- **Commitment:** Delivery by 18:00...

  [Chunk 3] 500 chars
  ### Standard (ST)
- **Commitment:** Delivery within 2 business days by 18:00.
- **Eligible zones:** All zones.
- **Surcharge:** Base rate, no surcharge.
- **Failure threshold:** SLA breach after 18:00...


── Strategy B — Recursive 500 (10 total chunks from this doc) ──
  [Chunk 1] 255 chars
  # Meridian Logistics — Service Level Agreement (SLA) Policies (Synthetic)

> **Disclaimer

In [9]:
# ── Decision: which strategy to use going forward? ────────────────────────────
#
# For this project we'll use Strategy B (Recursive, 500 chars, 50 overlap)
# as our working baseline because:
#   1. Respects natural language boundaries (paragraphs > sentences)
#   2. Medium size — good balance of precision vs context richness
#   3. The standard recommendation in production RAG systems
#
# We'll revisit chunk size in Phase 2 when we run RAGAS evaluation.

CHUNKS = recursive_chunks
print(f"✅ Using recursive chunks: {len(CHUNKS)} total chunks across all documents")

# Enrich metadata — add chunk index per document for traceability
doc_counters = {}
for chunk in CHUNKS:
    src = chunk.metadata["source"]
    doc_counters[src] = doc_counters.get(src, 0) + 1
    chunk.metadata["chunk_index"] = doc_counters[src]

print("\nChunks per document:")
for src, count in doc_counters.items():
    print(f"  • {Path(src).name:<35} {count} chunks")

✅ Using recursive chunks: 55 total chunks across all documents

Chunks per document:
  • capacity_planning.md                10 chunks
  • routing_rules.md                    11 chunks
  • zone_definitions.md                 9 chunks
  • station_operations.md               15 chunks
  • sla_policies.md                     10 chunks


---

## Step 3 — Embeddings

### Why this matters — what is an embedding?

An **embedding** is a mathematical representation of text as a list of numbers
(a vector). The key property: **semantically similar texts have similar vectors**.

For example:
- `"delivery zone rural"` → `[0.12, -0.43, 0.89, ...]`  (384 numbers)
- `"remote area dispatch"` → `[0.11, -0.41, 0.91, ...]`  (very close!)
- `"quarterly earnings report"` → `[-0.72, 0.34, -0.12, ...]`  (very different)

The **sentence-transformers** library provides models trained specifically to
make this property hold. `all-MiniLM-L6-v2` is:
- Fast (runs on CPU in milliseconds)
- Compact (384 dimensions)
- Good quality for English semantic search

### How transformers produce embeddings

1. Text is tokenised → sequence of token IDs
2. Token IDs pass through the transformer layers (attention + feed-forward)
3. The `[CLS]` token output (or mean pooling) gives a single vector for the whole sentence
4. That vector is your embedding — ready for similarity search

In [10]:
# ── Load the embedding model ───────────────────────────────────────────────────
# First time: this downloads ~90MB from Hugging Face. Cached afterwards.

print("Loading embedding model (downloads on first run)...")
t0 = time.time()

model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"✅ Model loaded in {time.time()-t0:.1f}s")
print(f"   Embedding dimensions: {model.get_sentence_embedding_dimension()}")

Loading embedding model (downloads on first run)...
✅ Model loaded in 2.6s
   Embedding dimensions: 384


In [11]:
# ── Experiment 3A: Understand what an embedding looks like ────────────────────

sample_sentences = [
    "What is the SLA for next-day delivery in Zone 3?",
    "Express delivery commitment for semi-rural areas",
    "How many stops can a van do per day in urban zones?",
    "Maximum stops per route for city centre delivery",
    "Quarterly financial earnings report for FY2024",  # unrelated — should score low
]

embeddings = model.encode(sample_sentences, normalize_embeddings=True)

print(f"Shape: {embeddings.shape}  ({len(sample_sentences)} sentences × {embeddings.shape[1]} dimensions)")
print(f"\nFirst embedding (first 10 values of 384):")
print(np.round(embeddings[0][:10], 4))

Shape: (5, 384)  (5 sentences × 384 dimensions)

First embedding (first 10 values of 384):
[-0.0435 -0.0531 -0.0168  0.0216  0.0135 -0.0351 -0.0372 -0.0349 -0.0358
  0.0371]


In [12]:
# ── Experiment 3B: Cosine similarity matrix ────────────────────────────────────
# Cosine similarity measures the angle between two vectors.
# Score = 1.0 → identical meaning; 0.0 → unrelated; negative → opposite

sim_matrix = cosine_similarity(embeddings)

print("Cosine similarity matrix (rounded to 2dp):")
print("(rows/cols = sentence index)\n")

labels = [s[:45] + "..." if len(s) > 45 else s for s in sample_sentences]

print(f"{'':>5}", end="")
for i in range(len(sample_sentences)):
    print(f"  [{i}] ", end="")
print()

for i, row in enumerate(sim_matrix):
    print(f"[{i}] ", end="")
    for val in row:
        marker = " ★" if val > 0.7 and val < 0.999 else "  "
        print(f"{val:>5.2f}{marker}", end=" ")
    print(f"  ← {labels[i][:40]}")

print("\n★ = high semantic similarity (>0.70)")
print("\n💡 Notice: sentences 0&1 (SLA / Zone 3) and 2&3 (stops per route) score")
print("   high with each other, but both score low against sentence 4 (earnings).")

Cosine similarity matrix (rounded to 2dp):
(rows/cols = sentence index)

       [0]   [1]   [2]   [3]   [4] 
[0]  1.00    0.36    0.33    0.34    0.03     ← What is the SLA for next-day delivery in
[1]  0.36    1.00    0.20    0.39   -0.03     ← Express delivery commitment for semi-rur
[2]  0.33    0.20    1.00    0.54   -0.01     ← How many stops can a van do per day in u
[3]  0.34    0.39    0.54    1.00   -0.04     ← Maximum stops per route for city centre 
[4]  0.03   -0.03   -0.01   -0.04    1.00     ← Quarterly financial earnings report for 

★ = high semantic similarity (>0.70)

💡 Notice: sentences 0&1 (SLA / Zone 3) and 2&3 (stops per route) score
   high with each other, but both score low against sentence 4 (earnings).


In [13]:
# ── Embed all chunks ───────────────────────────────────────────────────────────
# Now we embed every chunk in our knowledge base.
# This is the 'indexing' phase — done once, then stored.

print(f"Embedding {len(CHUNKS)} chunks...")
t0 = time.time()

texts = [chunk.page_content for chunk in CHUNKS]
chunk_embeddings = model.encode(
    texts,
    batch_size=32,              # process 32 at a time for memory efficiency
    normalize_embeddings=True,  # normalise → cosine sim = dot product (faster)
    show_progress_bar=True,
)

elapsed = time.time() - t0
print(f"\n✅ Embedded {len(chunk_embeddings)} chunks in {elapsed:.1f}s")
print(f"   Embedding matrix shape: {chunk_embeddings.shape}")
print(f"   Memory: ~{chunk_embeddings.nbytes / 1024:.1f} KB")

Embedding 55 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Embedded 55 chunks in 0.3s
   Embedding matrix shape: (55, 384)
   Memory: ~82.5 KB


---

## Step 4 — FAISS vector store (in-memory)

### Why FAISS first?

**FAISS (Facebook AI Similarity Search)** is a low-level library for
efficient similarity search. Unlike ChromaDB, it gives you direct control
over the index type, making it great for understanding what's actually happening:

- `IndexFlatL2` — exhaustive brute-force search (exact, slow for large sets)
- `IndexFlatIP` — inner product (= cosine similarity if vectors are normalised)
- `IndexHNSWFlat` — approximate search using a graph structure (fast, production-grade)

Starting with FAISS forces you to understand the index structure before
ChromaDB abstracts it away.

In [14]:
# ── Build FAISS index ──────────────────────────────────────────────────────────

DIM = chunk_embeddings.shape[1]  # 384 for all-MiniLM-L6-v2

# IndexFlatIP = Inner Product search.
# Since our embeddings are L2-normalised, inner product == cosine similarity.
faiss_index = faiss.IndexFlatIP(DIM)

# FAISS requires float32
vectors = chunk_embeddings.astype(np.float32)
faiss_index.add(vectors)

print(f"✅ FAISS index built")
print(f"   Index type: {type(faiss_index).__name__}")
print(f"   Vectors stored: {faiss_index.ntotal}")
print(f"   Dimensions: {faiss_index.d}")

✅ FAISS index built
   Index type: IndexFlatIP
   Vectors stored: 55
   Dimensions: 384


In [15]:
# ── Query the FAISS index ──────────────────────────────────────────────────────

def faiss_search(query: str, k: int = 4) -> list[dict]:
    """
    Embed a query and retrieve top-k most similar chunks from FAISS.
    Returns a list of dicts with chunk text, source, and similarity score.
    """
    query_vector = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = faiss_index.search(query_vector, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = CHUNKS[idx]
        results.append({
            "text": chunk.page_content,
            "source": Path(chunk.metadata["source"]).name,
            "chunk_index": chunk.metadata.get("chunk_index", "?"),
            "score": float(score),
        })
    return results


def print_results(query: str, results: list[dict]) -> None:
    print(f"\n{'='*65}")
    print(f"Query: {query}")
    print(f"{'='*65}")
    for i, r in enumerate(results):
        print(f"\n[{i+1}] Score: {r['score']:.4f}  |  Source: {r['source']} (chunk {r['chunk_index']})")
        print(f"    {r['text'][:280].strip()}...")


# ── Run test queries ───────────────────────────────────────────────────────────
queries = [
    "What is the SLA for next-day delivery in Zone 3?",
    "How many stops can a van complete per day in urban areas?",
    "When should we use vendor fleet instead of owned fleet?",
    "What happens after 3 failed delivery attempts?",
]

for q in queries:
    results = faiss_search(q, k=3)
    print_results(q, results)


Query: What is the SLA for next-day delivery in Zone 3?

[1] Score: 0.6572  |  Source: sla_policies.md (chunk 2)
    ### Priority Overnight (PO)
- **Commitment:** Delivery by 10:30 the next business day.
- **Eligible zones:** Zone 1 and Zone 2 only.
- **Surcharge:** +€4.50 per parcel vs Standard rate.
- **Failure threshold:** SLA breach if delivery occurs after 10:30.
- **Compensation:** Full r...

[2] Score: 0.6530  |  Source: zone_definitions.md (chunk 5)
    ### Zone 3 — Semi-Rural
- **Coverage:** Towns and villages between 20–60 km from the nearest hub.
- **Fleet type:** Large vans (up to 12t), occasional HGV trunking.
- **Max stops per route:** 60 stops/day.
- **Standard SLA:** Next-day delivery by 18:00 where volume justifies dail...

[3] Score: 0.6503  |  Source: sla_policies.md (chunk 4)
    ### Standard (ST)
- **Commitment:** Delivery within 2 business days by 18:00.
- **Eligible zones:** All zones.
- **Surcharge:** Base rate, no surcharge.
- **Failure threshold:** SLA breach

---

## Step 5 — ChromaDB vector store (persistent)

### Why migrate from FAISS to ChromaDB?

FAISS is fast and educational, but it has limitations for a real project:

| | FAISS | ChromaDB |
|---|---|---|
| **Persistence** | In-memory only (lost on restart) | Persists to disk automatically |
| **Metadata filtering** | Manual, no built-in support | Built-in `where` filter on any metadata field |
| **Document management** | No add/delete after creation | Add, update, delete documents |
| **LangChain integration** | Requires custom wrapper | Native `Chroma` class in `langchain-community` |
| **Embedding integration** | Manual encode step | Handles encoding internally |

For a production-style project, ChromaDB is the right choice.
You already understand what it's doing under the hood because you just did it with FAISS.

In [16]:
# ── Build ChromaDB collection ──────────────────────────────────────────────────

CHROMA_DIR = Path("../.chroma")  # persistent storage directory
COLLECTION_NAME = "lastmile_delivery_kb"

# Create a persistent client — data is saved to CHROMA_DIR automatically
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Delete collection if it exists (clean start for experimentation)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("Deleted existing collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # use cosine distance metric
)

print(f"✅ Created collection: {COLLECTION_NAME}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Deleted existing collection
✅ Created collection: lastmile_delivery_kb


In [17]:
# ── Add documents to ChromaDB ──────────────────────────────────────────────────
# ChromaDB needs: ids, documents (text), embeddings, and optionally metadatas

ids = [f"chunk_{i:04d}" for i in range(len(CHUNKS))]

metadatas = [
    {
        "source": Path(c.metadata["source"]).name,
        "chunk_index": c.metadata.get("chunk_index", 0),
    }
    for c in CHUNKS
]

t0 = time.time()
collection.add(
    ids=ids,
    documents=texts,
    embeddings=chunk_embeddings.tolist(),  # ChromaDB wants plain Python lists
    metadatas=metadatas,
)

print(f"✅ Added {collection.count()} chunks to ChromaDB in {time.time()-t0:.1f}s")
print(f"   Persisted to: {CHROMA_DIR.resolve()}")

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ Added 55 chunks to ChromaDB in 0.0s
   Persisted to: /Users/evertongomes/lastmile-delivery-rag/.chroma


In [18]:
# ── Query ChromaDB ─────────────────────────────────────────────────────────────

def chroma_search(query: str, k: int = 4, source_filter: str = None) -> list[dict]:
    """
    Query ChromaDB. Optionally filter by source document.
    The 'where' filter is a ChromaDB feature FAISS doesn't have.
    """
    query_embedding = model.encode([query], normalize_embeddings=True).tolist()

    where_clause = {"source": {"$eq": source_filter}} if source_filter else None

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        where=where_clause,
        include=["documents", "metadatas", "distances"],
    )

    output = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        output.append({
            "text": text,
            "source": meta["source"],
            "chunk_index": meta["chunk_index"],
            "distance": float(dist),  # lower = more similar (cosine distance)
        })
    return output


# ── Compare FAISS vs ChromaDB on the same queries ─────────────────────────────
test_query = "What is the failed delivery attempt policy?"

faiss_results = faiss_search(test_query, k=3)
chroma_results = chroma_search(test_query, k=3)

print(f"Query: {test_query}\n")
print("FAISS results:")
for r in faiss_results:
    print(f"  Score: {r['score']:.4f} | {r['source']} | {r['text'][:120].strip()}...")

print("\nChromaDB results:")
for r in chroma_results:
    print(f"  Dist:  {r['distance']:.4f} | {r['source']} | {r['text'][:120].strip()}...")

print("\n💡 Results should be identical (same embeddings, same chunks).")
print("   FAISS reports similarity score (higher = better).")
print("   ChromaDB reports cosine distance (lower = better).")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: What is the failed delivery attempt policy?

FAISS results:
  Score: 0.7773 | sla_policies.md | ---

## 3. Failed Delivery Handling

### Attempt Policy
- A maximum of **3 delivery attempts** are made before a parcel...
  Score: 0.5917 | sla_policies.md | | Service | Zone 1 | Zone 2 | Zone 3 | Zone 4 |
|---|---|---|---|---|
| Priority Overnight | ≥98.5% | ≥97.0% | N/A | N/A...
  Score: 0.5188 | sla_policies.md | ### Express (EX)
- **Commitment:** Delivery by 18:00 the next business day.
- **Eligible zones:** All zones (Zone 4 subj...

ChromaDB results:
  Dist:  0.2227 | sla_policies.md | ---

## 3. Failed Delivery Handling

### Attempt Policy
- A maximum of **3 delivery attempts** are made before a parcel...
  Dist:  0.4083 | sla_policies.md | | Service | Zone 1 | Zone 2 | Zone 3 | Zone 4 |
|---|---|---|---|---|
| Priority Overnight | ≥98.5% | ≥97.0% | N/A | N/A...
  Dist:  0.4812 | sla_policies.md | ### Express (EX)
- **Commitment:** Delivery by 18:00 the next business day.
- **

In [19]:
# ── Bonus: ChromaDB metadata filtering ────────────────────────────────────────
# This is a key ChromaDB advantage — you can restrict search to specific documents.
# Useful when you want to answer questions scoped to a single policy document.

print("=== Filtered query — SLA doc only ===")
filtered = chroma_search(
    query="What compensation do customers receive for late delivery?",
    k=3,
    source_filter="sla_policies.md",  # only search within this document
)
for r in filtered:
    print(f"  [{r['source']}] {r['text'][:200].strip()}...")

=== Filtered query — SLA doc only ===
  [sla_policies.md] ### Express (EX)
- **Commitment:** Delivery by 18:00 the next business day.
- **Eligible zones:** All zones (Zone 4 subject to volume availability).
- **Surcharge:** +€1.80 per parcel vs Standard rate...
  [sla_policies.md] ### Priority Overnight (PO)
- **Commitment:** Delivery by 10:30 the next business day.
- **Eligible zones:** Zone 1 and Zone 2 only.
- **Surcharge:** +€4.50 per parcel vs Standard rate.
- **Failure th...
  [sla_policies.md] | Service | Zone 1 | Zone 2 | Zone 3 | Zone 4 |
|---|---|---|---|---|
| Priority Overnight | ≥98.5% | ≥97.0% | N/A | N/A |
| Express | ≥97.5% | ≥96.0% | ≥94.0% | ≥90.0% |
| Standard | ≥99.0% | ≥98.5%...


---

## Step 6 — Latency benchmarking

### Why this matters

In a production RAG system, every millisecond counts. The retrieval step
must be fast enough that the user isn't waiting just for context lookup.
Let's benchmark both backends.

In [20]:
# ── Latency benchmark — FAISS vs ChromaDB ─────────────────────────────────────

BENCHMARK_QUERIES = [
    "SLA for Zone 2 express delivery",
    "vendor fleet Gold tier criteria",
    "morning sortation process steps",
    "peak season capacity buffer rate",
    "driver maximum hours per day regulation",
]

N_RUNS = 20  # run each query 20 times and average

def benchmark(search_fn, queries, n_runs):
    times = []
    for _ in range(n_runs):
        for q in queries:
            t0 = time.perf_counter()
            search_fn(q, k=4)
            times.append((time.perf_counter() - t0) * 1000)  # ms
    return np.mean(times), np.std(times), np.min(times), np.max(times)

faiss_mean, faiss_std, faiss_min, faiss_max = benchmark(faiss_search, BENCHMARK_QUERIES, N_RUNS)
chroma_mean, chroma_std, chroma_min, chroma_max = benchmark(chroma_search, BENCHMARK_QUERIES, N_RUNS)

print(f"{'Backend':<15} {'Mean (ms)':<12} {'Std':<10} {'Min':<10} {'Max':<10}")
print("-" * 57)
print(f"{'FAISS':<15} {faiss_mean:<12.2f} {faiss_std:<10.2f} {faiss_min:<10.2f} {faiss_max:<10.2f}")
print(f"{'ChromaDB':<15} {chroma_mean:<12.2f} {chroma_std:<10.2f} {chroma_min:<10.2f} {chroma_max:<10.2f}")
print("\n💡 The embedding step dominates latency for small indices.")
print("   At scale (millions of vectors), HNSW indexing becomes critical.")

Backend         Mean (ms)    Std        Min        Max       
---------------------------------------------------------
FAISS           7.27         8.50       5.41       78.64     
ChromaDB        6.74         0.93       6.20       13.14     

💡 The embedding step dominates latency for small indices.
   At scale (millions of vectors), HNSW indexing becomes critical.


---

## Step 7 — Save processed chunks for Phase 2

We save the chunks to disk so Phase 2 can load them without re-processing.

In [20]:
# ── Persist chunks as JSON ─────────────────────────────────────────────────────
# ChromaDB is already persisted. We also save a JSON version for inspection
# and for loading in Phase 2 without needing LangChain's document format.

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

chunks_serialisable = [
    {
        "id": ids[i],
        "text": c.page_content,
        "source": Path(c.metadata["source"]).name,
        "chunk_index": c.metadata.get("chunk_index", 0),
        "char_count": len(c.page_content),
    }
    for i, c in enumerate(CHUNKS)
]

output_path = PROCESSED_DIR / "chunks.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks_serialisable, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(chunks_serialisable)} chunks to {output_path}")

# Summary stats
char_counts = [c["char_count"] for c in chunks_serialisable]
print(f"\nChunk character count statistics:")
print(f"  Min:    {min(char_counts)} chars")
print(f"  Max:    {max(char_counts)} chars")
print(f"  Mean:   {np.mean(char_counts):.0f} chars")
print(f"  Median: {np.median(char_counts):.0f} chars")

✅ Saved 55 chunks to ../data/processed/chunks.json

Chunk character count statistics:
  Min:    53 chars
  Max:    497 chars
  Mean:   346 chars
  Median: 380 chars


---

## Phase 1 — Summary & key takeaways

| Step | What you did | Concept learned |
|---|---|---|
| 1 | Loaded 5 Markdown documents | `Document` objects, metadata importance |
| 2 | Compared 3 chunking strategies | Chunk size/overlap trade-offs, recursive splitting |
| 3 | Embedded chunks with sentence-transformers | Dense vectors, cosine similarity, transformer inference |
| 4 | Built FAISS index and ran queries | Vector indexing internals (IndexFlatIP) |
| 5 | Migrated to ChromaDB | Persistence, metadata filtering, production-readiness |
| 6 | Benchmarked retrieval latency | Performance characteristics of both backends |
| 7 | Saved processed chunks | Pipeline reproducibility |

### What's coming in Phase 2

We'll connect this retrieval system to a real LLM (Mistral via Ollama) to build
the full RAG chain:

```
User query → embed → retrieve top-k chunks → inject into prompt → LLM → answer
```

We'll also implement advanced retrieval techniques (HyDE, MMR, cross-encoder
re-ranking) and add multi-turn conversation memory.

---
*Built as part of a structured AI learning plan. See the project README for full context.*